In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA

In [2]:
# Load train and test datasets
train_data = pd.read_csv('/kaggle/input/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')

In [3]:
# Divide the training data into training and validation sets
train_x, val_x, train_y, val_y = train_test_split(train_data.drop('Survived', axis=1), train_data['Survived'], test_size=0.2, random_state=0)

In [4]:
# Build the pipeline
numerical_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [5]:
# Use the ColumnTransformer to apply the transformations to the appropriate columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, ['Age', 'Fare']),
        ('cat', categorical_transformer, ['Embarked', 'Sex'])
    ])

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(mutual_info_classif, k=5)),
    ('reductor', PCA(n_components=3)),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=0))
])

In [6]:
# Use GridSearchCV to find the best hyperparameters for the model
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(model_pipeline, param_grid, cv=5)
grid_search.fit(train_x, train_y)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         SimpleImputer(strategy='median'),
                                                                         ['Age',
                                                                          'Fare']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                           

In [7]:
# Get the best parameters and the best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_
print("Best parameters: ", best_params)
print("Best score: ", best_score)

Best parameters:  {'classifier__max_depth': 5, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
Best score:  0.7949473062149119


In [8]:
# Use the best parameters to fit the pipeline to the training data
model_pipeline.set_params(**best_params)
model_pipeline.fit(train_x, train_y)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['Age', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Embarked', 'Sex'])])),
                ('selector',
                 SelectKBest(k=5,
                             score_func=<function mutual_info_classif at 0x7f8756fb9560>)),
                ('reductor', PCA(n_components=3)), ('scaler', StandardScaler()),
                

In [9]:
# Get the accuracy of the model on the validation data
val_acc = model_pipeline.score(val_x, val_y)
print("Validation accuracy: ", val_acc)

Validation accuracy:  0.7932960893854749


In [10]:
# Predict the survival of the passengers in the test data
predictions = model_pipeline.predict(test_data)

In [11]:
# Store the predictions in a DataFrame and write it to a CSV file
submission = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': predictions})
submission.to_csv('submission.csv', index=False)

🚀 Was it helpful? thanks you for upvote :)

👉 [Check other of my models ](https://www.kaggle.com/roxket/code)